# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:
import pandas as pd
import numpy as np
import faiss
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
print('Kolom tersedia:', df.columns.tolist())
print('Kategori FAQ:', df['category'].unique().tolist())
df.head()

Total entri knowledge base: 59
Kolom tersedia: ['faq_id', 'category', 'question', 'answer']
Kategori FAQ: ['Pemesanan', 'Pengiriman', 'Pembayaran', 'Pengembalian', 'Akun', 'Promo', 'Produk', 'Customer Service']


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [2]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print(f'Total unit retrievable: {len(documents)}')
print()
print('Contoh unit retrievable:')
print(documents[0])

Total unit retrievable: 59

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [5]:

import os

INDEX_PATH = '../data/faq_faiss.index'

if os.path.exists(INDEX_PATH):
    # ── SKIP EMBED — load index dari disk ──
    print('File FAISS ditemukan, memuat dari disk...')
    index = faiss.read_index(INDEX_PATH)
    print('Index berhasil dimuat, total vektor:', index.ntotal)
    print(f'Verifikasi: {index.ntotal == len(df)} (harus True)')

    # Buat dummy embeddings shape agar kode selanjutnya tidak error
    # (dimensi diambil dari index yang sudah ada)
    print(f'Dimensi vektor: {index.d}')

else:
    # ── EMBED DARI AWAL ──
    def get_embedding(text):
        result = client.models.embed_content(
            model='gemini-embedding-001',
            contents=text
        )
        return result.embeddings[0].values

    print('File FAISS belum ada, membuat embeddings dari awal...')
    embeddings = [get_embedding(doc) for doc in documents]
    embeddings = np.array(embeddings).astype('float32')
    print('Shape embeddings:', embeddings.shape)
    print(f'Dimensi vektor: {embeddings.shape[1]}')
    print(f'Jumlah dokumen ter-embed: {embeddings.shape[0]}')

    # Buat dan simpan index FAISS
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    os.makedirs('../data', exist_ok=True)
    faiss.write_index(index, INDEX_PATH)
    print(f'Index FAISS disimpan ke: {INDEX_PATH}')
    print(f'Jumlah vektor tersimpan: {index.ntotal}')
    print(f'Verifikasi: {index.ntotal == len(df)} (harus True)')

File FAISS belum ada, membuat embeddings dari awal...
Shape embeddings: (59, 3072)
Dimensi vektor: 3072
Jumlah dokumen ter-embed: 59
Index FAISS disimpan ke: ../data/faq_faiss.index
Jumlah vektor tersimpan: 59
Verifikasi: True (harus True)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [6]:
import os

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)
print(f'Verifikasi: {index.ntotal == len(df)} (harus True — semua entri berhasil disimpan)')

# Simpan index ke disk
os.makedirs('../data', exist_ok=True)
faiss.write_index(index, '../data/faq_faiss.index')
print('Index FAISS berhasil disimpan ke: ../data/faq_faiss.index')

Jumlah vektor tersimpan di index: 59
Verifikasi: True (harus True — semua entri berhasil disimpan)
Index FAISS berhasil disimpan ke: ../data/faq_faiss.index


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [7]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
print('=== Contoh Semantic Search ===')
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(f"Rank {r['rank']} | Distance: {r['distance']:.4f} | [{r['category']}] {r['question']}")

=== Contoh Semantic Search ===
Rank 1 | Distance: 0.3335 | [Pengiriman] Bagaimana jika paket saya belum sampai melebihi estimasi?
Rank 2 | Distance: 0.4088 | [Pengiriman] Kenapa status pengiriman saya tidak berubah selama beberapa hari?
Rank 3 | Distance: 0.4320 | [Pengiriman] Kenapa nomor resi saya belum bisa dilacak di website kurir?


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [8]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ #{r['rank']} [{r['category']}]:\nPertanyaan: {r['question']}\nJawaban: {r['answer']}" 
        for r in search_results
    )
    prompt = f'''[PERSONA]
Kamu adalah asisten customer service e-commerce yang ramah, profesional, dan membantu.

[TASK]
Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ yang diberikan di bagian Context. 
Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut 
dan sarankan pengguna untuk menghubungi customer service secara langsung.
Jangan mengarang atau menambahkan informasi di luar FAQ yang diberikan.

[CONTEXT - FAQ Relevan]
{context}

[FORMAT]
- Jawab dalam Bahasa Indonesia yang ramah dan mudah dipahami.
- Gunakan poin-poin jika ada beberapa langkah atau informasi penting.
- Tutup jawaban dengan tawaran bantuan lebih lanjut.

[PERTANYAAN PENGGUNA]
{query}

[JAWABAN]'''
    return prompt

# Tampilkan contoh prompt
sample_results = semantic_search('cara bayar cicilan', top_k=2)
sample_prompt = build_prompt('cara bayar cicilan', sample_results)
print('=== Contoh Prompt yang Dibangun ===')
print(sample_prompt)

=== Contoh Prompt yang Dibangun ===
[PERSONA]
Kamu adalah asisten customer service e-commerce yang ramah, profesional, dan membantu.

[TASK]
Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ yang diberikan di bagian Context. 
Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut 
dan sarankan pengguna untuk menghubungi customer service secara langsung.
Jangan mengarang atau menambahkan informasi di luar FAQ yang diberikan.

[CONTEXT - FAQ Relevan]
FAQ #1 [Pembayaran]:
Pertanyaan: Apakah saya bisa membayar dengan cicilan?
Jawaban: Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank.

FAQ #2 [Pembayaran]:
Pertanyaan: Metode pembayaran apa saja yang tersedia?
Jawaban: Kami mendukung transfer bank, kartu kredit/debit, e-wallet, dan pembayaran melalui minimarket rekanan.

[FORMAT]
- Jawab dalam Bahasa Indonesia yang ramah dan mudah dipahami.
- Gunakan poin-poin jika ada beberapa 

## 7. Mengintegrasikan LLM

In [9]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',  # sesuaikan versi model
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

# Contoh integrasi penuh
print('=== Contoh Integrasi LLM ===')
answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print(f"  - [{s['category']}] {s['question']} (distance: {s['distance']:.4f})")

=== Contoh Integrasi LLM ===


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Jawaban LLM:
Halo! Tentu, saya akan bantu jawab pertanyaannya ya.

Terkait pembayaran dengan cicilan, berikut informasinya:
- Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu.
- Ada nominal minimal yang ditentukan oleh pihak bank untuk dapat menggunakan fitur ini.

Apakah ada hal lain seputar metode pembayaran atau informasi lainnya yang ingin Anda tanyakan? Saya siap membantu!

Sumber FAQ yang digunakan:
  - [Pembayaran] Apakah saya bisa membayar dengan cicilan? (distance: 0.2092)
  - [Pembayaran] Apakah bisa membayar menggunakan lebih dari satu metode sekaligus? (distance: 0.5486)
  - [Pembayaran] Metode pembayaran apa saja yang tersedia? (distance: 0.6107)


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [12]:
test_queries = [
    # Query 1: Jelas ada di FAQ — tentang lacak paket
    "Bagaimana cara melacak paket yang sedang dikirim?",

    # Query 2: Jelas ada di FAQ — tentang pembatalan pesanan
    "Bagaimana cara membatalkan pesanan saya?",

    # Query 3: Parafrase/sinonim — 'pengiriman terlambat' alih-alih 'paket melebihi estimasi'
    "Barang saya sudah lama belum nyampe, sudah lewat dari jadwal estimasi",

    # Query 4: Parafrase/sinonim — 'uang kembali' alih-alih 'refund'
    "Kalau saya kembalikan produk yang rusak, kapan uang saya bisa balik?",

    # Query 5: Tidak ada di FAQ — tentang fitur wishlist
    "Apakah ada fitur wishlist atau simpan produk favorit di platform ini?",
]

print('=' * 70)
for i, q in enumerate(test_queries, 1):
    print(f'\n[QUERY {i}] {q}')
    print('-' * 70)
    answer, sources = generate_answer(q)
    print('\nFAQ yang di-retrieve:')
    for s in sources:
        print(f"  Rank {s['rank']} | Distance: {s['distance']:.4f} | [{s['category']}] {s['question']}")
    print()
    print('Jawaban LLM:')
    print(answer)
    print('=' * 70)


[QUERY 1] Bagaimana cara melacak paket yang sedang dikirim?
----------------------------------------------------------------------

FAQ yang di-retrieve:
  Rank 1 | Distance: 0.2601 | [Pengiriman] Bagaimana cara melacak paket yang sedang dikirim?
  Rank 2 | Distance: 0.4310 | [Pengiriman] Kenapa nomor resi saya belum bisa dilacak di website kurir?
  Rank 3 | Distance: 0.4763 | [Pengiriman] Bagaimana jika paket saya belum sampai melebihi estimasi?

Jawaban LLM:
Halo! Tentu, saya akan bantu jelaskan ya.

Untuk melacak paket yang sedang dikirim, Anda dapat mengikuti langkah-langkah berikut:
- Nomor resi akan muncul di halaman **'Pesanan Saya'** setelah barang dikirim.
- Anda bisa menggunakan nomor resi tersebut untuk melacak posisi paket di website kurir terkait.

Ada lagi hal lain yang bisa saya bantu?

[QUERY 2] Bagaimana cara membatalkan pesanan saya?
----------------------------------------------------------------------

FAQ yang di-retrieve:
  Rank 1 | Distance: 0.2740 | [Pemesanan]

### Catatan Analisis Relevansi

Berikut adalah hasil analisis dari 5 query yang diuji pada sistem semantic search + LLM:

---

**Query 1 — "Bagaimana cara melacak paket yang sedang dikirim?"** *(Query langsung / kata kunci sama)*

- **Relevansi retrieval**: ✅ Sangat relevan. Kalimat query hampir identik dengan FAQ #10, sehingga distance sangat kecil dan rank 1 langsung menunjuk ke entri yang tepat.
- **Kualitas jawaban LLM**: ✅ Jawaban akurat — menyebut nomor resi di halaman 'Pesanan Saya' dan cara melacak di website kurir, persis sesuai FAQ.

---

**Query 2 — "Bagaimana cara membatalkan pesanan saya?"** *(Query langsung / kata kunci sama)*

- **Relevansi retrieval**: ✅ Sangat relevan. FAQ #3 tentang pembatalan pesanan di-retrieve sebagai rank teratas.
- **Kualitas jawaban LLM**: ✅ LLM menjelaskan prosedur dengan benar: melalui 'Pesanan Saya' selama status masih 'Diproses', dan bahwa pembatalan tidak bisa dilakukan setelah status 'Dikirim'.

---

**Query 3 — "Barang saya sudah lama belum nyampe, sudah lewat dari jadwal estimasi"** *(Parafrase — bahasa informal/sinonim)*

- **Relevansi retrieval**: ✅ Baik. Meski menggunakan bahasa informal dan tidak menggunakan kata 'paket' atau 'estimasi pengiriman' secara eksplisit, semantic search berhasil me-retrieve FAQ #12 (paket melebihi estimasi) dan FAQ #14 (status tidak berubah) yang sama-sama relevan.
- **Kualitas jawaban LLM**: ✅ Jawaban tepat — menyarankan menghubungi customer service dengan nomor resi jika sudah lebih dari 3 hari melebihi estimasi.
- **Catatan**: Ini membuktikan semantic search menangkap *makna* ('barang belum sampai / lewat jadwal'), bukan hanya kata kunci eksak.

---

**Query 4 — "Kalau saya kembalikan produk yang rusak, kapan uang saya bisa balik?"** *(Parafrase — 'uang balik' alih-alih 'refund')*

- **Relevansi retrieval**: ✅ Baik. Frasa 'uang saya bisa balik' berhasil dipetakan ke konsep 'refund/pengembalian dana'. FAQ #25 (durasi proses refund) di-retrieve sebagai entri paling relevan, didampingi FAQ #27 (tujuan dana refund) — keduanya memang paling sesuai untuk query ini.
- **Kualitas jawaban LLM**: ✅ LLM menjawab dengan lengkap: dana diproses 3-7 hari kerja setelah barang diterima dan diverifikasi penjual, dikembalikan ke metode pembayaran asal atau saldo akun.

---

**Query 5 — "Apakah ada fitur wishlist atau simpan produk favorit di platform ini?"** *(Query di luar knowledge base)*

- **Relevansi retrieval**: ⚠️ Kurang relevan. Karena tidak ada FAQ tentang wishlist, sistem me-retrieve entri terdekat secara semantik (mungkin FAQ tentang fitur produk atau akun) meski tidak benar-benar menjawab pertanyaan. Distance cenderung lebih besar dibanding query sebelumnya.
- **Kualitas jawaban LLM**: ✅ LLM berperilaku sesuai instruksi prompt — menyatakan bahwa informasi tentang fitur wishlist tidak tersedia di knowledge base, dan menyarankan pengguna untuk menghubungi customer service langsung. Ini menunjukkan guardrail prompt berjalan dengan baik.

---

**Ringkasan:**
- Semantic search bekerja sangat baik untuk query langsung maupun parafrase/sinonim dalam Bahasa Indonesia.
- Kemampuan menangkap makna (bukan hanya kata kunci) terbukti pada Query 3 dan 4 — keunggulan utama semantic search dibanding keyword search.
- Untuk query di luar knowledge base, LLM berhasil menolak mengarang jawaban berkat instruksi prompt yang eksplisit.
- **Keterbatasan**: Jika query sangat ambigu atau topik sangat jauh dari semua entri FAQ, hasil retrieval bisa kurang relevan. Sistem ini hanya sebaik knowledge base yang tersedia.